# P05 on PolyBench with a T4 GPU (Phase 3.2)

Runs on **Colab** or **Kaggle** (it works in `/content` or `/kaggle/working`, whichever exists).

- **Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Keep the
  tab open; if the session disconnects, reconnect and Run all again: the run resumes from the
  rows already in the CSV (unless the runtime was reset).
- **Kaggle:** File → Import Notebook (this file). In the right-hand panel, Session options:
  **Accelerator → GPU T4 x2** and **Internet on** (both need a phone-verified account). Then
  **Save Version → Save & Run All (Commit)**: it runs in the background (up to 12 hours) with
  the browser closed, and the CSV appears under the version's **Output**.

Installing the NVIDIA HPC SDK takes about 10 minutes; the benchmark run takes longer (30
kernels, 6 variants, 3 runs each at the LARGE size, plus two no-FMA builds of ludcmp and lu).

Variants per kernel: `seq` (nvc -O3, no OpenMP), `seq-mp` (the same source with the OpenMP
flags: what `-mp=gpu` alone does), `p05` (P05's own choices with the T4
machine model: GPU, host threads or sequential per loop), `p05-naive` (every loop P05 can
offload goes to the GPU), and both again as `-warm` (the device started before `main`, so
the timed kernel doesn't pay the ~0.16 s start-up). For ludcmp and lu, `seq-nofma` and
`p05-nofma` are built with `-Mnofma` (no fused multiply-adds), checked against each other.
Correctness: exact hex dumps against `seq`. `gpu_launches` counts kernel launches (0 = every guard chose the host).

**When it finishes, download `polybench-t4.csv` (Colab: the last cell offers it; Kaggle: the
version's Output) and give it to Claude**; it goes to `eval/polybench/results/`.

Generated by `eval/polybench/make_gpu_notebook.py` at P05 2859f4e (30
kernels, LARGE_DATASET, 80 GPU pragmas in the gated rewrites): edit that and `gpu_run.py`, not this
notebook.


In [ ]:
# Where to work: Colab's /content or Kaggle's /kaggle/working.
import base64, gzip, os
ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
os.makedirs(ROOT, exist_ok=True)
os.environ['P05_ROOT'] = ROOT  # the %%bash cells cd here
os.chdir(ROOT)
RUNNER = gzip.decompress(base64.b64decode("""
H4sIAAAAAAAC/51aaXPbRhL9zl8xC9eWgQ0BUbLj3dBhqhQdiat8qGQ5WymtCgsCQxErEEAwoESu
Sv99X/cMLgKUvdEHEcdMT08fr4/Bi78crFVxMI/TA5nei3xbLrP01ciyrItloKR45R1NxcXke5Gl
4iJLtj/LNFwenIjX3pF3KB7icikC8cvFF2GfZEkwF1evxyK9D4W7yme3+drxRiOBP0NX4JFfrFMv
3wrXzeciB8k5kcStKkJRyIciLqUS166LcUq8uqHLUN3TVA+/N6PRZ5ks3DBLyyBOZSRsVUZJPAeH
ydaZCnkfJAc13YNVcCd9WjbNSjnPsjtaW67mMlIiLsejII1ofy9VvfZYxKmmEtKWGlp++dqL8206
HwsVp6EU5VKKiuwoDNKXpQiTLNUvCplnKi6zYuuZzS2zBIvuLrYKIknSxZxRJO9lkuUrmZZ4Hi6x
Py1jt7ozv5o1Fwz9R2XpFDLGAge3QSmjgx/vZJHK5CcvhOD1atlDKsJlFoeQrA1tjcUJVFYuCxmA
o6wQSv6xxqJxkDiGVhrE97JLy4Uk4nAr+BUJWhZb8fn4/EwkWZZDmgIyENlikWRBZLZEtjEaXchC
aErCfn98+cuZf3p8dfz57Gos3E+vHOZf/iHqP5qYFfFtnAYJ7CkTn3KZfrjQw2BaA8O0nOiJHisW
SXCL3WJyXgS3q0DBNh6WQVmZ5kh0/wLWXJRBRGXGhJaZwpYyqIeMJC6VoJ2VcXoLu43TEhRyeEb9
x+KvNDuu+DDy0INdlh0N1hdmsHn5EBSrcTOM73fZJMZUsAJ97Xww1VVebkUZFLeyBMHbGJKH74i5
XGQFWQzM2Q526cB/VFmsQxgo2Tst5YUQEdEv4xU2YhQGAUIyt9BgHmwVm+kOKVgtLAtmpsBE6a5z
LDfP1qWYeIdvhKosgaDh46ers8/iaHL0xj2cuIdvnPEusYdlHBKm0JoEASmJO8hzWB5ED1LMRQb3
M9aQZotVYKRGl7sEybBIf6Sq+TpOSuNRerRLZsJWslgrbHq1Tso4T7ZuEEWqzx1hjIBY6+laSOpt
s76IFXxNhnegFtxC+KocYEmP9cSVTBIlzj8ck0LKIgh5j0EOQYpFka0gCThpAjOPZAFR9NS4DKAb
YSfrKIRfRPFiIQsl5ltx5L2W7g9dkR+6R/9wRr+SXRvnn4pPHy78j18++Fe/Xp4dn34WM+ytQgYN
6lNxJO4BGIrQ/CQrChmWqVSYG4iLT+9///ns48mv/ukXEDq+vDz+/TPLGQiwEBJwJe6DIg6AaDk2
wN4jN9jnaCk32qEAgmG2wpYhMdYN5ANrsi0eZxE+kf0kZOGKTDyBJdwDLXmzQGbpQJDGL9j0w2yd
wl0pLhkrToI1EJzg7+NvJ9jyu/dnl/7xyYkP8bw7//3tiIKWiSATsZIBYo8GuI5jga14YTsEpkrW
IAGxXGYPSmAHZKoyjbATAyInn38TAbsNDCdOY7V8yzolNtVdnCMU8NSEJL4dYVwB31YZBkWxglGk
kDbI6fCqIHb2b6nWK+zmgYYT8qoyw8KRR5F7FK/yDPYDxiFUJat7RM/qMlPVVVG/Vst1GSf13Xqe
FxkCRj1SbdVodP7u7D0bybWlJWuNhWUUTJfGcugSaFCu+YrwxK+v+CLUZmS1PcxaBRsf2vVlUdAY
itqV3ngya8I3cM5UMKJ1G0mkBIl1M3oBeF2xGNdFWEPxmFGO8KkFfhReXPI/ttnRb8eX744/XvEO
bQuGSHQpxuD3PEiU7P44Y8GjEFNaA6+KdW9cx3FtC3DBW6SQ8c0zdFigedXF0DwzlvbaX6I1Yf8K
9dzuMs/NrSFtr7xods1cPXRQADz0ZvTP48sP/gk0AZt+gYQLGCfFj0AKb/nTyPeDsizi+bqUvm/b
LY06DgWiMg7FfRYz7vu0H5vuHPEIxgFDYgO6k7ejF9p8BKhWnr4KcrvMCHynYkPp0EZ8NxOHb0dP
7FwjZGkLQhk7XEVjgax59hGZw5ijJqIe33FKI4TEIlEclnamPAyMiyx19AtvnUfYuk1JN/Dt8Uk/
L4vttJZsgdmNG3r1kmGQl+tC+lgsx3padKXcVJfEUosf8+s0dCWmA0M8fUEZzhh3SKMxzFzBBTWj
m1DmbTTwrjS5s00eA7Cnu1TdQ2i1cnWMqwRmEN4u5GIsbrPSSAgSfckw/3K8D+dFg/PIacoHKXU+
ka6RxRfKI6UQqRhawUKzGZHv8WWCyRhJyYTfIWeYQ8CY4ilkF6XtMF/VTUUykakdOOIvM76aO33C
JuaCNEcz24rThaXnP2QFNjKr16TMYTMWW8q4/hvnNvHQoojlNsR/ywZMqoawuW7yro6VMF0QXWyx
EHPgke0iutobZ7zzZNtYgdHsb0GylmdFkRVdkr01+us09Nt0v4H2N0rPiGSxIekvdthpXmzIcMDV
wKD/Y6G2uhCHbL4ei2CubKzggrQjDviNfuSYd1tcHEr31WTiaGJmPdsqs0QWAazWImY17R9nNPgH
gZRRNiyBBL+Gq4RJoJTgKltvhXzH95E4lIA5hcp3THFdtayGHnoB2Kbn3ac5WTiwJw/KpQd26dem
YXjjdIeCgbt9g+ldMxxDqKiO4kLZ9VSgziZWpZ/dMQjtEgf+toj/B9VTe6qlq4+WNjIEbLueSiMs
x+NSydZRYWcBgF9BIri+qZ+/gNjefby6uLr0r36/OPN9Ts2QD5Hz6bL4dh0UEXL3X05OuEQ4SQKq
7QBrckHFdozMriIH+JvL5/bAA/qb4MedDVgxV/eYbiJSbTJ44dg7XDuTt+LpX2nXJSgUXLNqwnAs
qiXczDI34jtheQS+N8715IY8Y9J1jK7QLPd0Z9UZlV9IpGoTNHUO77hle5yu8z77csnn4IlSyriM
dQLHXZRVUNz5CWzFQqBGZrzocmZkcZ0gChWASOf6aHrDwJmQ4haMydW7FoNLpJ3SWORY3BEvY11o
tNhFqKBCK99SbUIxRG8LpmCm0xL0nL0BFi4podgKmwlN6wqmJtjUMM5bk23qMoDCXZwgb08hZVML
wI5TqiloEep+1ZGLNVKEg9ZFUqyegiHKa23endNYRPScXS6jwgWbDEK8DQM+eG41pOeBkkwbfDgd
W0uzsh4VK7BgR05XZW1AiLqISnpb0H4xhnRO02mFPkwTniNHihSZlG15S2tgEAcm2mzf4go4wsJx
PCo/bGdwZiWAYboV7RIk8iQIpf3S+uvEO0oWwkJygpsAF87O2+7LQcJ9ZiNitYMJ/ewsakybKxNj
2LquMebNm4ah6cLGmDsVNEiuZzpPn36bkczjdMgY9hjecCyI9sYAuekhZ6S3AqCyLTdao3zatU+r
tcwL8e+6ALDuYBv/BoIFBRf03BBjzzONzsZtF2ALXmt8sUUt0CDQ8s0aAB5QyFId3SRf+XZQchXg
dKAGirUeaWNP7uOgaz21gFwX3B6toO2XrlpeLRfKoLMFOekw76FqAL2y15kC/J8ONmN6gm2Pu3r3
4cxqYqbumc6EjUU/vbJuSD1mXcjQ59dVgkxkqWpiotc3TsUiR5W+IzSU2DyZVpuU7p1VxBopoOJh
hszkMCSmNJ/faRF9RyJ6B/N9Ti08otfN+1rMcrRO/v+JlA9UxwaUE9yM+uK4rvObG04QKUdqCROb
SrgI57AOF7ppYQSMxacNUwGjy8JufhAOxHwDLLpYBddVENVaSJDwKKgCgH1t6f7dIqAIRjmEe3iz
i09giAq9BqVQdRqMglW0cId4fLR2uozWFDV6UVmEaRo5T+09gEqXf3nNVK7o/ODK/3R+/v7T8Sls
dFa5RtNp73JqKlDTKLIZwzRpKpqrcMKP2+GD3lDNV0gP2WAUJIldWAP9glIGK9I4ETOGG35tKkph
PJQJBcjuVMM0YDw0fHOqaBgOcp3nc1/POy5u13RWdEF3iMtShdAodZBnvh9loe93dItU0DFUvCCK
/MBMBwIjGwEbhfxjTUW96SMsZZLPrP6pXw2w1l5ygLN99Bp45uRRn1lx+q2PnPYTpVgFqpBJsE7K
mcX3e0eH6r49uHZHOjSjd/snhu156X24fyhUyQcIqj3DnC8JF/9mYfj375+Zry22Pfv07Pz4y/ur
PliZfJNE2HcDTrRah1+mI6v2L23iSHvpzsnc/plNi7fc5nKGjLihcbR3Fp3mDk55tX8h00EamvVm
MqkMSskwQ94ocmTWWGU/46aGsSj1QO00s/62f2wtwG8Z3JwkPau24UMmyo6bk0bbbHAq3A86JJJm
rb0ZqyADHeu44S4WuVsdI81gW6hJ9Nmcpn0nZa74HBNp0vhZovoULlaC62DcJdIcb+qDpoNkbY5U
6HywEuxXBDQg/0ah15amTBEvWVsVUhWcj4AkIx4RVU1vjivgJp0gfim6NWFjd8CMsk0tV06Leima
qakRE+ko09QiDQTs+TN51a74zSTqwTCAMnXHREyOwql84PDFzUcCyX4TuKmwNW/qfqhqJnIUZ+2i
Ppe5Af5e10czSCnIkLj2BA3vNA7LS06V7EUr7vLJNDPF5UO7BXwOG/iYlefZOo12enp5oHTjiQ59
Z11uoU5q9IMoRaGZVXVGMa7i459UAhU2N5/1+VKtYUxriUIXSzrDs40gaVec69HO5l7Vp2glIO22
FodDBFu7p3lT2iBGtpMQ3oWhyTHqjuthLNV+85V2LXelxeMTVR7twybh/mSOOqkAGrVr5u7xVavE
M0fMqagOqnpdUT2iz6t2g0GOB7lm0w04xWyd7uyk6nxCtssBL1cBqInsK9lhpv8W+Sqt9o2MYRX7
TpeQDhFl+/+TtHTnpVuhzzmh1zZBi2lNUG9eH26N9qFAhwpvlSu7WpN7zK4LLyXUHVKmV6WsVDl2
c/nsoTpa0hqdQRpGqDNtPCZMz5iJ6kC/XbMdjvdhWvegdabZqU9aZ2HeZeaFaH0UwFqgg/fq5Nbr
Ogd9GYF4VRTj5kx+xidGE2/CFUX3IAE7STJC7rmn+yBf74CYdFO3Qfi/M+r1gQasg87iaB2qZKiG
GepR9aqS1uRra/DbAi5RrMO+zexWcddU5Jnju/S+OcDTGtw9xav+OLnnKNJJ9vvjGmmr9co+bHqq
moLpqzZhz8T3ky+nxybCW06fLEEX7e9fqaWNPhkgbBERq0KAZFCygzVrx2xomSonNDPgkO6hNueF
JTdxKR6L8KkvapkQbhE2zPZhg0HqazYYUlkHlLuEKhDG+Gf4ZTOnONccfRrqYybufBXSvhHB/pyX
mMP259yEFDboKiYUA4ZsBiFcjIX+4GOmuweu6R5QhkdfZczA3/X0aDK5GTAiaNRbJGu1HLBb7rnb
C+vxbnr45klwd216+P2TaDcppuKR6R8eTW6e+IgPxHa6kM+KkD9MqbbAB0n0HcOd1etm+6x3+uhK
wzvVNQOdag36w87dIMy3+vjX3KNm+8/6Bwd7hIi73hsWjKe/arL1yanKuq0jmxtEO2p9zkL0D/XW
OGzMtLc8l2GL1qdBM9gCfqbeq9sn3qcGB9M1fp5M+5OiWY2I7aj4NQr6WybiYQWwY+k4U+/NQrPC
999OCM4iDGyC4MbQ4a8E+CSKqT9PxniXxctXdjBj29V8NI2+axfuN91V1D7f2+t3j3qV6T+eBkvI
jmR2ZKLt52UapC+d6eHEe714UkI8sgKnP+yjVyvssboacnHaR5hkSlJlMIrp1JxY9n0Wh+9TG833
Le1Auqc2+h+8dHt3/i4AAA==
""")).decode()
open(os.path.join(ROOT, 'gpu_run.py'), 'w').write(RUNNER)
print('working in', ROOT)


In [ ]:
%%bash
nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv
nproc; grep -m1 'model name' /proc/cpuinfo


In [ ]:
%%bash
# NVIDIA HPC SDK (nvc -mp=gpu): the compiler the GPU evaluation uses (NOTES.md, 2026-10-02).
set -e
if ls /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc >/dev/null 2>&1; then echo "already installed"; exit 0; fi
curl -s https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK \
  | gpg --dearmor --yes -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' \
  > /etc/apt/sources.list.d/nvhpc.list
apt-get update -qq
PKG=$(apt-cache search --names-only '^nvhpc-[0-9]+-[0-9]+$' | awk '{print $1}' | sort -V | tail -1)
echo "Installing $PKG"; apt-get install -y -qq $PKG > /tmp/nvhpc.log 2>&1 || { tail -20 /tmp/nvhpc.log; exit 1; }
ls -d /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc


In [ ]:
%%bash
# PolyBench/C 4.2.1 at the commit eval/llm_baseline/fetch_polybench.sh pins.
set -e
cd "$P05_ROOT"
[ -d polybench/.git ] || git clone -q https://github.com/MatthiasJReisinger/PolyBenchC-4.2.1 polybench
git -C polybench checkout -q 3e872547cef7e5c9909422ef1e6af03cf4e56072
test "$(git -C polybench rev-parse HEAD)" = 3e872547cef7e5c9909422ef1e6af03cf4e56072 && echo "polybench at 3e872547cef7e5c9909422ef1e6af03cf4e56072"


In [ ]:
# P05's rewrites (made with -machine machine/colab-t4.json), packed by
# make_gpu_notebook.py (gzip + base64: one long line freezes the editor).
import base64, gzip, json, os
ROOT = os.environ['P05_ROOT']
PACKED = gzip.decompress(base64.b64decode("""
...
""")).decode()
FILES = json.loads(PACKED)
for p, text in FILES.items():
    os.makedirs(os.path.dirname(os.path.join(ROOT, 'rewrites', p)), exist_ok=True)
    open(os.path.join(ROOT, 'rewrites', p), 'w').write(text)
print(len(FILES), 'files')


In [ ]:
%%bash
cd "$P05_ROOT"
NVC=$(ls -d /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc | sort -V | tail -1)
CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
$NVC --version | grep -v '^$' | head -1
python3 gpu_run.py --pb polybench --src rewrites --cc $NVC --omp-flags "-mp=gpu -gpu=cc$CC" \
  --threads $(nproc) --runs 3 --dataset LARGE_DATASET --csv polybench-t4.csv


In [ ]:
import csv, math, os
CSV = os.path.join(os.environ['P05_ROOT'], 'polybench-t4.csv')
rows = list(csv.DictReader(open(CSV)))
t = {(r['kernel'], r['variant']): float(r['time_s']) for r in rows if r['status'] == 'ok' and r['time_s']}
bad = [(r['kernel'], r['variant'], r['status'], r['correct']) for r in rows
       if r['status'] != 'ok' or r['correct'] not in ('', 'exact', 'tolerance')]
print('not ok / not correct:', bad or 'none')
for v in ('seq-mp', 'p05', 'p05-naive', 'p05-warm', 'p05-naive-warm', 'p05-nofma'):
    s = 'seq-nofma' if v == 'p05-nofma' else 'seq'
    xs = [t[(k, s)] / t[(k, v)] for (k, vv) in t if vv == v and (k, s) in t]
    if xs:
        print(f"{v:15} geomean speedup over {s} {math.exp(sum(map(math.log, xs)) / len(xs)):.2f} ({len(xs)} kernels)")
try:
    from google.colab import files
    files.download(CSV)
except ImportError:  # Kaggle: files in /kaggle/working are the version's Output
    from IPython.display import FileLink, display
    print('CSV at', CSV)
    display(FileLink(os.path.relpath(CSV)))
